# RoPE: позиция как поворот

**RoFormer: Enhanced Transformer with Rotary Position Embedding** · Su et al. · 2021

[Оригинальная статья](https://arxiv.org/abs/2104.09864)

**Цель урока:** Написать rotary embedding и проверить сохранение нормы и зависимость dot product от относительного смещения.

## Перед началом

Разбор Attention, тригонометрия поворота и формы тензоров.

Около 40–60 минут с заданиями. CPU, PyTorch и Matplotlib; без скачивания весов и датасетов. Время — ориентир для чтения и работы, а не benchmark выполнения.

## Задача и идея

RoPE кодирует позицию, поворачивая пары координат Q и K. Это отличается от прибавления position embedding к токену. Разные пары используют разные частоты, поэтому близкие и далёкие сдвиги выражаются через несколько масштабов. V в обычном применении RoPE не вращается.

Важно различать математическое свойство относительной позиции и способность обученной модели работать на большем контексте. Первое проверяется точно, второе требует эксперимента; смена частот и context scaling не гарантируют качество.

## Граница воспроизведения

Реализовано вращение и его свойства, не RoFormer LM и не метод расширения контекста.

Код написан для этого урока по математическим идеям статьи. Это не скопированный авторский репозиторий. Успех assertions подтверждает указанные свойства реализации, но не результаты статьи.

## Формула → тензоры

Для пары `(a,b)` угол `phi = position * theta_i`: `(a*cos(phi)-b*sin(phi), a*sin(phi)+b*cos(phi))`. Частота `theta_i = base^(-2i/D)`.

`R(m)q · R(n)k = q · R(n-m)k`. Поэтому общий сдвиг m и n не меняет dot product. Ширина D должна быть чётной. Здесь используется соседнее разбиение на пары; другие layouts допустимы только при согласованной перестановке весов и координат.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Шаг 1

Считаем вращение явно в float64, чтобы численная проверка была точной.

In [ ]:
def rope(x, positions, base=10000.):
    width = x.shape[-1]
    assert width % 2 == 0
    frequency = base ** (-torch.arange(0, width, 2, dtype=x.dtype) / width)
    angle = positions.to(x.dtype)[..., None] * frequency
    even, odd = x[..., 0::2], x[..., 1::2]
    result = torch.stack([even * angle.cos() - odd * angle.sin(),
                          even * angle.sin() + odd * angle.cos()], dim=-1)
    return result.flatten(-2)
x = torch.randn(8, dtype=torch.float64)
rotated = rope(x, torch.tensor(7))
print('norm error:', (x.norm() - rotated.norm()).abs().item())
assert torch.allclose(x.norm(), rotated.norm(), atol=1e-12)

## Шаг 2

Проверяем относительный сдвиг и одновременно сдвигаем обе позиции.

In [ ]:
q = torch.randn(8, dtype=torch.float64); k = torch.randn_like(q)
m, n, offset = 5, 11, 20
left = rope(q, torch.tensor(m)) @ rope(k, torch.tensor(n))
right = q @ rope(k, torch.tensor(n - m))
shifted = rope(q, torch.tensor(m + offset)) @ rope(k, torch.tensor(n + offset))
print('relative error:', abs(left - right).item(), 'common-shift error:', abs(left - shifted).item())
assert torch.allclose(left, right, atol=1e-12)
assert torch.allclose(left, shifted, atol=1e-12)

## Шаг 3

Смотрим, как dot product меняется с расстоянием; проверяем differentiability.

In [ ]:
positions = torch.arange(128)
scores = torch.stack([rope(q, torch.tensor(0)) @ rope(k, p) for p in positions])
plt.plot(positions, scores, color='#d73535'); plt.xlabel('relative offset'); plt.ylabel('Q dot K'); plt.show()
differentiable = torch.randn(2, 6, 8, requires_grad=True)
result = rope(differentiable, torch.arange(6))
result.mul(torch.randn_like(result)).sum().backward()
assert torch.isfinite(differentiable.grad).all()
print('gradient norm:', differentiable.grad.norm().item())

## Как читать результат

Одна пара Q/K может давать осциллирующую кривую, а не монотонное затухание. Тест нормы проверяет вращение; тест общего сдвига проверяет позиционную структуру. Оба нужны: сохранение нормы само по себе ещё не подтверждает корректность RoPE.

## Самостоятельные эксперименты

1. Поверните только Q: сохранится ли общий сдвиг?
2. Смените base с 10000 на 100.
3. Передайте D=7 и объясните ошибку.

Сначала запишите прогноз, затем меняйте по одному фактору. Для честного сравнения сбрасывайте ядро и выполняйте все ячейки с начала. Сохраните копию для своих изменений.

## Проверка рассуждений — после эксперимента

Если K не вращается согласованно, dot product зависит от абсолютного положения Q. Base меняет сетку частот. Нечётная ширина не разбивается на полные пары; partial rotary требует явно выделить чётную подчасть.

## Что записать в лабораторный журнал

- Гипотеза и изменённый параметр.
- Seed, размер данных, число шагов и используемая precision.
- Значения проверок до и после, график и объяснение расхождения с прогнозом.
- Что этот эксперимент **не** позволяет заключить.

Связанный раздел доступен по кнопке «Открыть раздел студии» над уроком.